In [0]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

from pyspark.sql import functions as F
from nsw_property import parser

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "nsw_property")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing/sales/"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_sales_lines"
SILVER = f"{CATALOG}.{SCHEMA}.silver_property_sales"
QUARANTINE = f"{CATALOG}.{SCHEMA}.quarantine_property_sales"
FILE_CHECKS = f"{CATALOG}.{SCHEMA}.file_reconciliation"


print("Landing: ", LANDING)
print("Bronze: ", BRONZE)

In [0]:
new_lines = (
    spark.read.text(LANDING + "*.DAT")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

if spark.catalog.tableExists(BRONZE):
    already_loaded = spark.table(BRONZE).select("_source_file").distinct()
    new_lines = new_lines.join(already_loaded, "_source_file", "left_anti")

new_count = new_lines.count()
new_lines.write.mode("append").saveAsTable(BRONZE)

print(f"Shineer nemsen mur: {new_count}")

In [0]:
display(spark.sql(f"""
                  SELECT 
                  _source_file,
                  count(*) as line_count,
                  MIN(_ingested_at) as ingested_at
                  FROM {BRONZE}
                  GROUP BY _source_file"""))

In [0]:
records = parser.split_records(spark.table(BRONZE))

checks = parser.reconcile(records)
checks.write.mode("overwrite").saveAsTable(FILE_CHECKS)
display(checks)

complete_files = checks.where("is_complete").select("_source_file")

In [0]:
sales = parser.add_quality_flags(
    parser.build_sales(records.join(complete_files, "_source_file"))
)

display(sales.groupBy("is_valid").count())

In [0]:
# Davhardliig arilgah
from pyspark.sql.window import Window

latest_first = (
    Window
    .partitionBy("property_id", "dealing_number")
    .orderBy(F.desc("download_ts"))
)

valid = (
    sales
    .where("is_valid")
    .withColumn("_rank", F.row_number().over(latest_first))
    .where("_rank == 1")
    .drop("_rank", "dq_issues", "is_valid")
    .withColumn("_updated_at", F.current_timestamp())
)



In [0]:
if not spark.catalog.tableExists(SILVER):
    valid.limit(0).write.saveAsTable(SILVER)

valid.createOrReplaceTempView("silver_updates")

spark.sql(f"""
          MERGE INTO {SILVER} AS target
          USING silver_updates as source
          ON target.property_id = source.property_id 
          AND target.dealing_number = source.dealing_number
          WHEN MATCHED AND source.download_ts > target.download_ts
          THEN UPDATE SET *
          WHEN NOT MATCHED
          THEN INSERT *
          """)

In [0]:
(
    sales
    .where("NOT is_valid")
    .withColumn("_quarintined_at", F.current_timestamp())
    .write.mode("overwrite")
    .saveAsTable(QUARANTINE)
)

In [0]:
print("Silver: ", spark.table(SILVER).count())
print("Quarantine: ", spark.table(QUARANTINE).count())

display(
    spark.sql(f"DESCRIBE HISTORY {SILVER}")
    .select("version", "timestamp", "operation", "operationMetrics")
)